<a href="https://colab.research.google.com/github/tlh222k/ai-from-scratch/blob/main/phases/03-deep-learning-core/framework/01-getting-started-with-pytorch/chapter-1-pytorch-fundamentals-and-setup/hands-on-practical/03-interacting-with-NumPy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Kiểm tra cài đặt PyTorch (Verifying Your PyTorch Installation)**

- Trước tiên, hãy đảm bảo PyTorch đã được cài đặt chính xác và có thể sử dụng trong môi trường Python của bạn. Mở trình thông dịch Python hoặc Jupyter Notebook, sau đó chạy các lệnh sau:

In [35]:
import torch

# Pytorch
print(f"Pytorch Version: {torch.__version__}")


# Kiểm tra xem CUDA (hỗ trợ GPU) có khả dụng hay không
if torch.cuda.is_available():
    print(f"CUDA (GPU) khả dụng. Device: {torch.cuda.get_device_name(0)}")
    device = torch.device("cuda")
else:
    print("CUDA (GPU) không khả dụng")
    device = torch.device("cpu")

print(f"Default device: {device}")

Pytorch Version: 2.11.0+cu130
CUDA (GPU) khả dụng. Device: Tesla T4
Default device: cuda


- Việc thực thi đoạn mã này xác nhận rằng thư viện **torch** có thể được nhập vào chương trình và cung cấp thông tin về phiên bản đang sử dụng.

- Đoạn mã cũng kiểm tra khả năng sử dụng **GPU**, điều này rất quan trọng vì để tăng tốc các phép tính sau này.

## **Tương tác với NumPy (Interacting with NumPy)**

- PyTorch tích hợp tốt với NumPy. Hãy thực hành chuyển đổi qua lại giữa **mảng NumPy (NumPy array)** và **Tensor** của PyTorch.

### **Chuyển mảng NumPy thành Tensor PyTorch (NumPy Array to PyTorch Tensor)**

In [36]:
import numpy as np

# In phiên bản PyTorch
print(f"NumPy Version: {np.__version__}")

numpy_array = np.array([[1.0, 2.0],
                        [3.0, 4.0]])
print("\nNumPy Array:")
print(numpy_array)
print(f"Type: {type(numpy_array)}")

# Chuyển đổi sang tensor PyTorch
tensor_from_numpy = torch.from_numpy(numpy_array)
print("\nTensor from NumPy array:")
print(tensor_from_numpy)
print(f"Type: {type(tensor_from_numpy)}")

# Quan trọng: Trên CPU, torch.from_numpy chia sẻ bộ nhớ với mảng NumPy
# Việc thay đổi một đối tượng sẽ ảnh hưởng đến đối tượng còn lại
numpy_array[0, 0] = 99.0
print("\nNumPy array after modification:")
print(numpy_array)
print("Tensor after modifying NumPy array (shared memory):")
print(tensor_from_numpy)

NumPy Version: 2.1.3

NumPy Array:
[[1. 2.]
 [3. 4.]]
Type: <class 'numpy.ndarray'>

Tensor from NumPy array:
tensor([[1., 2.],
        [3., 4.]], dtype=torch.float64)
Type: <class 'torch.Tensor'>

NumPy array after modification:
[[99.  2.]
 [ 3.  4.]]
Tensor after modifying NumPy array (shared memory):
tensor([[99.,  2.],
        [ 3.,  4.]], dtype=torch.float64)


### **Chuyển Tensor PyTorch thành mảng NumPy (PyTorch Tensor to NumPy Array)**

In [39]:
# Sử dụng một tensor khác để tránh ảnh hưởng từ thay đổi trước đó
another_tensor = torch.tensor([[5, 6], [7, 8]], dtype=torch.float64)
print("\nAnother PyTorch Tensor:")
print(another_tensor)
print(50*"-")

# Chuyển đổi thành mảng NumPy
numpy_from_tensor = another_tensor.numpy()
print("\nNumPy array from Tensor:")
print(numpy_from_tensor)
print(f"Type: {type(numpy_from_tensor)}")
print(50*"-")

# Một lần nữa, bộ nhớ được chia sẻ khi sử dụng CPU
another_tensor[1, 1] = 100.0
print("\nTensor after modification:")
print(another_tensor)
print("NumPy array after modifying Tensor (shared memory):")
print(numpy_from_tensor)


Another PyTorch Tensor:
tensor([[5., 6.],
        [7., 8.]], dtype=torch.float64)
--------------------------------------------------
Default device: cuda

NumPy array from Tensor:
[[5. 6.]
 [7. 8.]]
Type: <class 'numpy.ndarray'>
--------------------------------------------------

Tensor after modification:
tensor([[  5.,   6.],
        [  7., 100.]], dtype=torch.float64)
NumPy array after modifying Tensor (shared memory):
[[  5.   6.]
 [  7. 100.]]


- Cơ chế chia sẻ bộ nhớ giữa mảng NumPy và Tensor trên CPU giúp sử dụng bộ nhớ hiệu quả, nhưng cần được xử lý cẩn thận vì có thể xảy ra những thay đổi ngoài ý muốn.

In [38]:
import torch

# 1. Tạo một tensor trực tiếp trên GPU
gpu_tensor = torch.tensor([[5, 6], [7, 8]], dtype=torch.float64, device='cuda')
print(f"Thiết bị của gpu_tensor: {gpu_tensor.device}")

# 2. Thử chuyển trực tiếp sang NumPy (Dòng này sẽ gây ra lỗi nếu không xử lý)
try:
    numpy_from_gpu = gpu_tensor.numpy()
except TypeError as e:
    print(f"\n[LỖI THỰC TẾ]: {e}")

# 3. Cách giải quyết đúng: Phải đưa về CPU trước rồi mới chuyển sang NumPy
numpy_from_gpu_correct = gpu_tensor.cpu().numpy()
print("\nChuyển đổi thành công sau khi đưa về CPU:")
print(numpy_from_gpu_correct)

Thiết bị của gpu_tensor: cuda:0

[LỖI THỰC TẾ]: can't convert cuda:0 device type tensor to numpy. Use Tensor.cpu() to copy the tensor to host memory first.

Chuyển đổi thành công sau khi đưa về CPU:
[[5. 6.]
 [7. 8.]]


### **Giải thích cơ chế chuyển đổi Tensor trên GPU sang NumPy**

- **Hạn chế của NumPy**: Thư viện NumPy chỉ hoạt động và quản lý vùng nhớ trên **CPU (Bộ nhớ hệ thống)**. Nó không có khả năng trực tiếp đọc hay ghi đè lên bộ nhớ của **GPU (VRAM)**.
- **Lý do xảy ra lỗi**: Khi bạn khởi tạo một Tensor với tham số `device='cuda'`, dữ liệu của Tensor đó hoàn toàn nằm trên bộ nhớ VRAM của GPU. Do đó, việc gọi trực tiếp `.numpy()` sẽ thất bại vì PyTorch không thể ánh xạ trực tiếp vùng nhớ GPU sang mảng NumPy trên CPU.
- **Giải pháp**:
  1. Sử dụng `.cpu()` để sao chép (copy) dữ liệu từ GPU trở lại CPU (RAM hệ thống).
  2. Sau khi dữ liệu đã ở trên CPU, ta gọi tiếp `.numpy()` để chuyển đổi một cách an toàn.

*Lưu ý: Khi thực hiện đưa về CPU rồi mới chuyển sang NumPy, mối liên kết chia sẻ bộ nhớ (shared memory) trước đó sẽ không còn hoạt động giống như việc chuyển đổi trực tiếp hoàn toàn trên CPU.*